# VISCERA · Run all → download model weights

**No code edits required.** Choose **Runtime → Run all** and approve Google Drive
access when Google asks. The dataset ZIP stays in the same Drive folder as this
notebook: **VISCERA Final Training 2026-10-07**. The notebook verifies it, installs
an isolated environment, trains, validates, freezes, audits, checks inference and
downloads **one `viscera-final-weights.zip`**. A copy remains in Drive.

A GPU is requested. Core models also train on CPU. The optional pinned Qwen LoRA
experiment runs only with CUDA; skipped/failed adapter status is shown explicitly
and does not prevent exporting the verified core models. No API keys are needed.
The first isolated PyTorch installation may take several minutes.

**Dataset:** 1,536 geometry recipes / 24 templates; 768 appearance recipes;
9,216 compiler-derived language examples; 3,600 disease recipes / 25 profiles /
12 organs; 200 realized disease surfaces; 3 CC BY pancreatic clinical cases.
These counts describe separate datasets, not independent patients or one large
paired cohort. The clinical prior has only one training patient.

**What the models learn:** bounded geometry/material/disease controls, decoded
by the supplied anatomy library. Validation may select a stronger regression or
empirical baseline instead of a neural candidate. Optional language weights are
a guarded text-to-spec adapter, not an unrestricted text-to-3D foundation model.
There are no measured physics labels or calibrated organ mechanics in this ZIP.

Keep this notebook paired with its exact dataset archive. Notebook code has been
reviewed and locally exercised; hosted GPU availability remains controlled by
Google. [Colab documentation](https://research.google.com/colaboratory/faq.html)


In [ ]:
"""Standard-library-only bootstrap embedded verbatim into the Colab notebook.

No project module is imported before the pinned ZIP and complete file manifest
are verified. Training happens in a separate interpreter, never the live kernel.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath


def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def safe_member(name: str) -> PurePosixPath:
    path = PurePosixPath(name)
    if (
        not name
        or "\\" in name
        or path.is_absolute()
        or ":" in name
        or any(part in ("", ".", "..") for part in name.split("/"))
        or path.parts[0] != "viscera"
        or len(path.parts) < 2
    ):
        raise ValueError(f"Unsafe archive member: {name!r}")
    return path


def verify_tree(root: Path, manifest: dict) -> None:
    if root.is_symlink():
        raise ValueError("Extracted bundle root must not be a symlink")
    hashes = manifest.get("artifact_hashes")
    if not isinstance(hashes, dict) or not hashes:
        raise ValueError("Bundle manifest has no file hashes")
    actual_files = set()
    for path in root.rglob("*"):
        if path.is_symlink():
            raise ValueError("Symlinks are forbidden inside the extracted bundle")
        if path.is_file():
            actual_files.add(path.relative_to(root).as_posix())
    if actual_files != set(hashes) | {"final-bundle-manifest.json"}:
        raise ValueError(
            "Extracted bundle has missing or unlisted files; use a fresh work directory"
        )
    for relative, expected in hashes.items():
        safe_member("viscera/" + relative)
        path = root / relative
        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
            raise ValueError(f"Escaped manifest member: {relative}")
        if not path.is_file() or file_digest(path) != expected:
            raise ValueError(f"Corrupt or missing bundle file: {relative}")


def extract_verified(
    archive_path: Path, expected_sha256: str, destination: Path
) -> tuple[Path, dict]:
    if not re.fullmatch(r"[a-f0-9]{64}", expected_sha256):
        raise ValueError("This notebook must be built with a final dataset SHA-256")
    if file_digest(archive_path) != expected_sha256:
        raise ValueError(
            "Dataset ZIP does not match this notebook. Use the matching release files."
        )
    destination.mkdir(parents=True, exist_ok=True)
    root = destination / "viscera"
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        names = [item.filename for item in entries]
        if len(names) != len(set(names)) or len(entries) > 25000:
            raise ValueError("Duplicate or excessive ZIP entries")
        if sum(item.file_size for item in entries) > 2_000_000_000:
            raise ValueError("Dataset archive exceeds release size limit")
        for item in entries:
            safe_member(item.filename)
            mode = item.external_attr >> 16
            if item.is_dir() or stat.S_IFMT(mode) not in (0, stat.S_IFREG):
                raise ValueError("Only regular-file ZIP entries are accepted")
            if item.flag_bits & 1:
                raise ValueError("Encrypted ZIP entries are not supported")
        manifest_name = "viscera/final-bundle-manifest.json"
        manifest = json.loads(archive.read(manifest_name))
        if manifest.get("format") != "viscera-final-colab-bundle-v1":
            raise ValueError("Unsupported final dataset bundle format")
        required = {
            "shape",
            "material",
            "language",
            "disease_recipes",
            "disease_geometry",
            "clinical",
        }
        if set(manifest.get("dataset_paths", {})) != required:
            raise ValueError("Incomplete dataset lane contract")
        for relative in manifest["dataset_paths"].values():
            safe_member("viscera/" + relative)
        expected_names = {"viscera/" + name for name in manifest["artifact_hashes"]} | {
            manifest_name
        }
        if set(names) != expected_names:
            raise ValueError("Archive and manifest members differ")
        if root.exists():
            verify_tree(root, manifest)
            actual_manifest = json.loads((root / "final-bundle-manifest.json").read_text())
            if actual_manifest != manifest:
                raise ValueError("Existing extraction belongs to a different bundle")
            return root, manifest
        temporary = Path(tempfile.mkdtemp(prefix=".extract-", dir=destination))
        try:
            for item in entries:
                target = temporary.joinpath(*PurePosixPath(item.filename).parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output, 1024 * 1024)
            verify_tree(temporary / "viscera", manifest)
            (temporary / "viscera").rename(root)
        finally:
            shutil.rmtree(temporary)
    return root, manifest


def install_environment(root: Path, environment: Path) -> Path:
    """Use a clean venv; retain the host's working PyTorch/CUDA wheel variant."""
    override = os.environ.get("VISCERA_TRAIN_PYTHON")
    if override:
        python = Path(override).resolve()
        if not python.is_file():
            raise ValueError("VISCERA_TRAIN_PYTHON is not a Python executable")
    else:
        python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
        if not python.exists():
            created = subprocess.run([sys.executable, "-m", "venv", str(environment)])
            if created.returncode:
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "virtualenv==20.35.3"], check=True
                )
                subprocess.run([sys.executable, "-m", "virtualenv", str(environment)], check=True)
        probe = subprocess.run(
            [
                sys.executable,
                "-c",
                "import torch,json; print(json.dumps({'version':torch.__version__.split('+')[0], 'cuda':torch.version.cuda, 'available':torch.cuda.is_available()}))",
            ],
            capture_output=True,
            text=True,
        )
        host = (
            json.loads(probe.stdout.strip())
            if probe.returncode == 0
            else {"version": "2.14.1", "cuda": None, "available": False}
        )
        version = host["version"]
        if not re.fullmatch(r"2\.\d+\.\d+", version) or tuple(map(int, version.split(".")[:2])) < (
            2,
            6,
        ):
            version = "2.14.1"
            host = {"cuda": None, "available": False}
        variant = "cpu"
        if host["available"] and host["cuda"]:
            cuda = host["cuda"].split(".")
            if len(cuda) != 2 or not all(part.isdigit() for part in cuda):
                raise ValueError("Unsupported host CUDA version")
            variant = "cu" + "".join(cuda)
        command = [
            str(python),
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            f"torch=={version}",
        ]
        if sys.platform == "linux":
            command += ["--index-url", "https://download.pytorch.org/whl/" + variant]
        print(
            f"Installing isolated PyTorch {version} ({variant}); first setup can take several minutes.",
            flush=True,
        )
        subprocess.run(command, check=True)
    subprocess.run(
        [
            str(python),
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-r",
            str(root / "notebooks/requirements-final-colab.txt"),
        ],
        check=True,
    )
    # Import both training stacks in a fresh process; no live-kernel package state.
    subprocess.run(
        [
            str(python),
            "-c",
            "import numpy, scipy, torch, trimesh, pydantic, nibabel; from transformers import Qwen2ForCausalLM; from peft import LoraConfig; print('Training dependencies import successfully; CUDA:', torch.cuda.is_available())",
        ],
        check=True,
    )
    return python


def verify_weights(archive_path: Path, completion: dict) -> str:
    """Check the precise export produced by the successful training command."""
    actual = file_digest(archive_path)
    if completion.get("status") != "complete" or actual != completion.get("weights_sha256"):
        raise ValueError("Weights ZIP changed or training did not complete")
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or archive.testzip() is not None:
            raise ValueError("Export has duplicate entries or a CRC error")
        manifest_name = "viscera-weights/manifest.json"
        payload = archive.read(manifest_name)
        if hashlib.sha256(payload).hexdigest() != completion.get("export_manifest_sha256"):
            raise ValueError("Export manifest changed after verification")
        manifest = json.loads(payload)
        if manifest.get("format") != "viscera-final-training-v1":
            raise ValueError("Unknown weights format")
        hashes = manifest.get("artifact_hashes", {})
        if not hashes or set(names) != {manifest_name} | {
            "viscera-weights/" + key for key in hashes
        }:
            raise ValueError("Export and manifest members differ")
        for relative, expected in hashes.items():
            safe_member("viscera/" + relative)
            if hashlib.sha256(archive.read("viscera-weights/" + relative)).hexdigest() != expected:
                raise ValueError(f"Weights member checksum failed: {relative}")
    return actual


ARCHIVE_NAME = 'viscera-final-dataset-2026-10-07.zip'
EXPECTED_SHA256 = '6f0deb9ef6a44c3713d34f5f9eb8168fa121d1aa17ea2aaf289a0bb9fb824ed3'
DRIVE_FOLDER_NAME = "VISCERA Final Training 2026-10-07"
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
DRIVE_FOLDER = None
explicit_archive = os.environ.get("VISCERA_BUNDLE_PATH")
if explicit_archive:
    ARCHIVE = Path(explicit_archive).expanduser().resolve()
elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
    ARCHIVE = DRIVE_FOLDER / ARCHIVE_NAME
else:
    ARCHIVE = Path.cwd() / ARCHIVE_NAME
if not ARCHIVE.is_file():
    raise FileNotFoundError(f"Keep {ARCHIVE_NAME} in {DRIVE_FOLDER_NAME}, then Run all again. Looked at {ARCHIVE}")
WORK = Path(os.environ.get("VISCERA_WORK_DIR", "/content/viscera-final" if IS_COLAB else str(Path.cwd() / "viscera-final-work"))).resolve()
LOCAL_ZIP = WORK / ARCHIVE_NAME
WORK.mkdir(parents=True, exist_ok=True)
if ARCHIVE.resolve() != LOCAL_ZIP.resolve():
    if not LOCAL_ZIP.exists() or file_digest(LOCAL_ZIP) != EXPECTED_SHA256:
        shutil.copy2(ARCHIVE, LOCAL_ZIP)
ROOT, MANIFEST = extract_verified(LOCAL_ZIP, EXPECTED_SHA256, WORK / EXPECTED_SHA256[:12])
print("Verified dataset ZIP:", EXPECTED_SHA256)
print("Verified packaged files:", len(MANIFEST["artifact_hashes"]))
print("Dataset paths:", json.dumps(MANIFEST["dataset_paths"], indent=2))


## Install and verify the training environment

Training uses a separate Python environment so package installation does not
require restarting this notebook. Data are extracted to local runtime storage;
completed stages and output weights are retained in the dedicated Drive folder.


In [ ]:
TRAIN_PYTHON = install_environment(ROOT, WORK / "train-env")
runtime_probe = subprocess.check_output([str(TRAIN_PYTHON), "-c", "import torch,sys,json; print(json.dumps({'python':sys.version.split()[0],'torch':torch.__version__,'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"], text=True)
RUNTIME = json.loads(runtime_probe)
PROFILE = os.environ.get("VISCERA_RUN_PROFILE", "full")
DEVICE = os.environ.get("VISCERA_RUN_DEVICE", "auto")
LANGUAGE = os.environ.get("VISCERA_RUN_LANGUAGE", "auto")
runtime_id = hashlib.sha256(json.dumps(RUNTIME, sort_keys=True).encode()).hexdigest()[:8]
run_name = EXPECTED_SHA256[:12] + "-" + PROFILE + "-" + runtime_id
output_base = DRIVE_FOLDER / "runs" if DRIVE_FOLDER else WORK / "runs"
OUT = Path(os.environ.get("VISCERA_RUN_OUT", str(output_base / run_name)))
OUT.mkdir(parents=True, exist_ok=True)
print("Runtime:", json.dumps(RUNTIME, indent=2))
print("Run:", OUT)
print("Core training runs on CPU or CUDA. Optional language adapter requires CUDA in auto mode.")


## Train → select → freeze → audit → verify inference

This executes all stages with fixed defaults. Training and selection use the
training/validation splits; held-out evaluation follows model freezing. Preflight
checks the integrity and structure of all splits without fitting on audit targets.
A verified completed stage is reused when
you run the notebook again; an interrupted stage restarts in a new attempt folder.
The final ZIP includes selected inference models, manifests,
source/data identities, measurements, attribution and scope limits.


In [ ]:
command = [str(TRAIN_PYTHON), "-u", str(ROOT / "scripts/final_train.py"),
           "--root", str(ROOT), "--out", str(OUT), "--profile", PROFILE,
           "--device", DEVICE, "--language", LANGUAGE]
training_env = dict(os.environ, PYTHONPATH=str(ROOT / "py") + os.pathsep + str(ROOT), PYTHONDONTWRITEBYTECODE="1")
log_path = OUT / "notebook-training.log"
with log_path.open("a") as log:
    process = subprocess.Popen(command, cwd=ROOT, env=training_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    result = process.wait()
if result:
    raise RuntimeError(f"Training stopped (exit {result}); inspect {log_path}. No success is reported for a failed run.")
COMPLETION = json.loads((OUT / "completion.json").read_text())
METRICS = json.loads((OUT / "export/metrics.json").read_text())
print(json.dumps(METRICS, indent=2))


## Download and import into the Studio

The next cell checks the exported archive and starts a single browser download.
If your browser blocks automatic downloads, use the displayed file link or the
saved Drive copy. In the updated Studio, open **Trained models**, select this ZIP,
then choose **Validate & activate**. The server verifies compatibility before
activation. Use the learned sampling controls to generate new geometry, surfaces
or disease recipes. Clinical case retrieval and the optional, explicitly enabled
text adapter remain separately labeled.


In [ ]:
WEIGHTS = OUT / "viscera-final-weights.zip"
if not WEIGHTS.is_file():
    raise FileNotFoundError("The verified weights export is missing")
weights_sha = verify_weights(WEIGHTS, COMPLETION)
receipt = {"weights_file": WEIGHTS.name, "weights_sha256": weights_sha,
           "dataset_sha256": EXPECTED_SHA256, "runtime": RUNTIME,
           "profile": PROFILE, "bytes": WEIGHTS.stat().st_size}
receipt_path = OUT / "download-receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2) + "\n")
if DRIVE_FOLDER:
    delivered = DRIVE_FOLDER / "weights"
    delivered.mkdir(exist_ok=True)
    for path in (WEIGHTS, receipt_path):
        target = delivered / path.name
        temporary = target.with_suffix(target.suffix + ".tmp")
        shutil.copy2(path, temporary)
        temporary.replace(target)
    print("Saved Drive copy:", delivered)
print("Weights:", WEIGHTS)
print("SHA-256:", weights_sha)
from IPython.display import FileLink, display
display(FileLink(str(WEIGHTS)))
if IS_COLAB:
    from google.colab import files
    files.download(str(WEIGHTS))
print("Finished. Review metrics.json for the selected model and language status.")
